Github URL: [follow link](https://github.com/Sophiacyt/Wildfire-Wetland_BunkerAndersonTseng.git)

In [1]:
# import packages
import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd

First look at wetlands data - California wetlands from the National Wetlands Inventory

In [2]:
# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

Box folder found at /Users/katanderson/Library/CloudStorage/Box-Box


In [3]:
# create path object that goes to the specific folder on Box drive
# uploading wetlands that are bigger than a hectare
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands' / 'wetlands.shp'
print(wetlands_path.exists())

True


In [4]:
#read in shp, be PATIENT (took MB 4.5 mins)
wetlands_shp = gpd.read_file(wetlands_path)




In [5]:
# look at wetlands info and report shape

wetlands_shp.info()

# add a column for area in hectares

wetlands_shp["Hectares"] = wetlands_shp["Shape_Area"]/10000

#no datetime objects, yes geometry (this creates the overall polygon for each one)
wetlands_shp.shape

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 891086 entries, 0 to 891085
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   OBJECTID    891086 non-null  float64 
 1   ATTRIBUTE   891086 non-null  str     
 2   WETLAND_TY  891086 non-null  str     
 3   QAQC_CODE   22260 non-null   str     
 4   ACRES       891086 non-null  float64 
 5   NWI_ID      891086 non-null  str     
 6   Shape_Leng  891086 non-null  float64 
 7   Shape_Area  891086 non-null  float64 
 8   geometry    891086 non-null  geometry
dtypes: float64(4), geometry(1), str(4)
memory usage: 61.2 MB


(891086, 10)

Further subset wetlands data to include wetland units larger than 1 hectare (10000m^2)

In [6]:
# create wetlands_hectare to only inclue wetlands greater than 1 hectare
wetlands_hectare = wetlands_shp.loc[lambda df: df["Hectares"] > 1]

wetlands_hectare.shape

#calculate subset percent
subset = 179139/891086*100

print("we are using", subset, "pecrent of the wetlands polygons from the California National Wetlands Inventory Dataset")

we are using 20.103446805358853 pecrent of the wetlands polygons from the California National Wetlands Inventory Dataset


Next look at fire perimeter data from CalFire - fire perimeters from 1910 to 2025, although we likely want to subset the data.

In [ ]:
# fire data path is different because it is not large, so stored in github

project_fldr = Path.cwd().parent

raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'

# read in CSV
fire_perims = pd.read_csv(raw_fldr / 'California_Historic_Fire_Perimeters_4486510518992991706.csv')

# convert alarm date, containment date to datetime objects
# want year to not include a day/month so will convert to string for now
fire_perims["Year"] = fire_perims["Year"].astype("Int64")

# alarm date and containment date have date/time included
fire_perims["Alarm Date"] = pd.to_datetime(fire_perims["Alarm Date"], format="%m/%d/%Y %I:%M:%S %p")
fire_perims["Containment Date"] = pd.to_datetime(fire_perims["Containment Date"], format="%m/%d/%Y %I:%M:%S %p")
fire_perims.info()

# report number of rows and columns 
fire_perims.shape


/Users/katanderson/Documents/MEM/Wildfire-Wetland_BunkerAndersonTseng/scripts


FileNotFoundError: [Errno 2] No such file or directory: '/Users/katanderson/Documents/MEM/Wildfire-Wetland_BunkerAndersonTseng/scripts/data/raw/California_Historic_Fire_Perimeters_4486510518992991706.csv'

In [6]:
# subset fire data to look at one year, 2025
fire_perims_25 = fire_perims[fire_perims['Year'] == 2025]
fire_perims_25.head()

# see if there are na values
fire_perims_25.isna().sum()

OBJECTID                          0
Year                              0
State                             0
Agency                            0
Unit ID                           0
Fire Name                         0
Local Incident Number             0
IRWIN ID                          0
Alarm Date                        0
Containment Date                  1
Cause                             0
Collection Method                 0
Management Objective              0
GIS Calculated Acres              0
Comments                        197
Complex Name                    231
Complex ID                      231
Fire Number (historical use)    243
GlobalID                          0
Decade                            0
Shape__Area                       0
Shape__Length                     0
dtype: int64